In [8]:

import six
import os
import glob
import numpy as np
import shutil
import subprocess
import torch
root_dir = '/data/dataset/trek150/TREK-150'



class Trek150Dataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, split: str = 'train'):
        self.root_dir = root_dir
        self.anno_files = sorted(glob.glob(os.path.join(self.root_dir, '*/groundtruth_rect.txt')))
        self.seq_names = [f.split('/')[-2] for f in self.anno_files]
        self.seq_dirs = [os.path.join(self.root_dir, n) for n in self.seq_names]
        
    def __getitem__(self, index):
        """
        Args:
            index (integer or string): Index or name of a sequence.

        Returns:
            tuple: (img_files, anno), where ``img_files`` is a list of
                file names and ``anno`` is a N x 4 (rectangles) numpy array.
        """
        if isinstance(index, six.string_types):
            if not index in self.seq_names:
                raise Exception('Sequence {} not found.'.format(index))
            index = self.seq_names.index(index)

        img_files = sorted(glob.glob(
            os.path.join(self.seq_dirs[index], 'img/*.jpg')))

        # load annotations
        anno = np.loadtxt(self.anno_files[index], delimiter=',')
        assert len(img_files) == len(anno)
        assert anno.shape[1] == 4

        return img_files, anno

    def __len__(self):
        return len(self.seq_names)

In [16]:
data = Trek150Dataset(root_dir)
data.seq_names
len(data)
data.seq_dirs

['/data/dataset/trek150/TREK-150/P03-P03_02-558',
 '/data/dataset/trek150/TREK-150/P03-P03_02-56',
 '/data/dataset/trek150/TREK-150/P03-P03_02-612',
 '/data/dataset/trek150/TREK-150/P03-P03_04-406',
 '/data/dataset/trek150/TREK-150/P03-P03_04-48',
 '/data/dataset/trek150/TREK-150/P03-P03_04-57',
 '/data/dataset/trek150/TREK-150/P03-P03_05-52',
 '/data/dataset/trek150/TREK-150/P03-P03_07-608',
 '/data/dataset/trek150/TREK-150/P03-P03_09-4',
 '/data/dataset/trek150/TREK-150/P03-P03_09-411',
 '/data/dataset/trek150/TREK-150/P03-P03_09-413',
 '/data/dataset/trek150/TREK-150/P03-P03_09-414',
 '/data/dataset/trek150/TREK-150/P03-P03_09-417',
 '/data/dataset/trek150/TREK-150/P03-P03_09-419',
 '/data/dataset/trek150/TREK-150/P03-P03_09-629',
 '/data/dataset/trek150/TREK-150/P03-P03_09-631',
 '/data/dataset/trek150/TREK-150/P03-P03_14-645',
 '/data/dataset/trek150/TREK-150/P03-P03_14-690',
 '/data/dataset/trek150/TREK-150/P03-P03_19-432',
 '/data/dataset/trek150/TREK-150/P03-P03_19-433',
 '/dat

In [14]:
data[0][1]

array([[ 447.,  394.,  635.,  537.],
       [ 466.,  383.,  675.,  552.],
       [ 494.,  339.,  689.,  585.],
       [ 517.,  314.,  697.,  610.],
       [ 546.,  301.,  697.,  613.],
       [ 578.,  290.,  685.,  640.],
       [ 612.,  287.,  674.,  655.],
       [ 641.,  285.,  654.,  666.],
       [ 659.,  284.,  646.,  674.],
       [ 676.,  282.,  636.,  681.],
       [ 688.,  281.,  631.,  689.],
       [ 692.,  280.,  628.,  687.],
       [ 688.,  274.,  632.,  689.],
       [ 687.,  265.,  634.,  701.],
       [ 690.,  256.,  632.,  705.],
       [ 684.,  248.,  638.,  709.],
       [ 679.,  239.,  644.,  713.],
       [ 674.,  230.,  649.,  717.],
       [ 669.,  221.,  655.,  721.],
       [ 663.,  212.,  661.,  725.],
       [ 658.,  204.,  667.,  729.],
       [ 653.,  195.,  673.,  734.],
       [ 649.,  187.,  679.,  738.],
       [ 646.,  182.,  683.,  740.],
       [ 643.,  177.,  687.,  742.],
       [ 643.,  173.,  688.,  744.],
       [ 644.,  170.,  688.,  745.],
 

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

import torch
import numpy as np
import pandas as pd
from PIL import Image
%cd /data/joohyun7u/project/vq2d-lightning
%load_ext autoreload
%autoreload 2

# from ltvu.structures import ResponseTrack

p_clips_dir = Path("/data/dataset/trek150/TREK-150")
p_pred_pt = Path("outputs/batch/2025-02-18/84365/trek150/intermediate_predictions.pt")
preds = torch.load(p_pred_pt, weights_only=True)
dfs_gt, dfs_pred, pred_frames = [], [], []
for clip_uid, clip_preds in preds.items():
    class_name = clip_uid.split('-')[0]
    df_gt = pd.read_csv(p_clips_dir / clip_uid / 'groundtruth_rect.txt', header=None, names=['x', 'y', 'w', 'h'])
    df_pred = pd.DataFrame(clip_preds['ret_bboxes'], columns=['x1', 'y1', 'x2', 'y2'])
    
    with open(p_clips_dir / clip_uid / 'frames.txt', 'r') as f:
        frame_idxs = [line.strip() for line in f.readlines()]
    p_frames = [Path(p_clips_dir / clip_uid / f'img/frame_{str(idx).zfill(10)}.jpg') for idx in frame_idxs]
    pred_frames.append(p_frames)
    # pred_frames.append([Image.open(p) for p in p_frames])
    dfs_gt.append(df_gt)
    dfs_pred.append(df_pred)


/data/joohyun7u/anaconda3/envs/vqlight3/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/data/joohyun7u/project/vq2d-lightning
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
dfs_pred

[              x1           y1           x2           y2
 0     460.618225   358.575317  1079.617310   970.765320
 1     479.468384   359.586304  1116.760498   965.515686
 2     519.981201   337.262756  1164.078247   947.884705
 3     545.175415   308.843658  1199.908936   972.327454
 4     567.586792   299.036865  1228.073486   983.171997
 ..           ...          ...          ...          ...
 156  1419.339966   946.833557  1920.000000  1070.984619
 157  1427.416626   977.875000  1902.528198  1073.833984
 158  1422.421509   992.017151  1853.989868  1074.810303
 159  1430.378418  1007.338623  1876.201294  1077.570801
 160  1430.542114   636.336609  1912.618774  1080.000000
 
 [161 rows x 4 columns],
              x1          y1           x2          y2
 0    850.668884  197.608459  1191.710083  490.681335
 1    853.507202  198.111481  1191.831909  480.539642
 2    857.659363  194.933395  1192.467285  457.777283
 3    856.374634  193.532532  1191.881348  457.115143
 4    821.088196  1

In [8]:
dfs_gt

[        x     y    w    h
 0     447   394  635  537
 1     466   383  675  552
 2     494   339  689  585
 3     517   314  697  610
 4     546   301  697  613
 ..    ...   ...  ...  ...
 156  1444   979  306  100
 157  1487  1027  278   52
 158  1485  1034  284   45
 159  1519  1068  254   11
 160  1519  1068  254   11
 
 [161 rows x 4 columns],
        x    y    w    h
 0    835  200  367  271
 1    833  197  364  274
 2    837  194  362  277
 3    836  194  365  277
 4    830  191  366  279
 ..   ...  ...  ...  ...
 246    7  111  448  456
 247   12  115  458  468
 248   16  119  468  480
 249   28  123  470  493
 250   41  127  471  505
 
 [251 rows x 4 columns],
        x     y    w    h
 0    800   398  363  345
 1    802   397  361  346
 2    803   395  359  347
 3    804   393  357  349
 4    806   392  359  350
 ..   ...   ...  ...  ...
 233  606  1001  188   79
 234  614  1030  156   49
 235  623  1036  123   43
 236  631  1057   91   22
 237  631  1057   91   22
 
 [238 ro

In [ ]:
import cv2
import numpy as np
from tqdm import tqdm
from PIL import Image, ImageDraw
from pathlib import Path

output_dir = Path("outputs/batch/2025-02-18/84365/trek150/tracking_result")
output_dir.mkdir(parents=True, exist_ok=True)  # Ensure output directory exists

fps = 60  # Frame rate
frame_size = (1280, 720)  # Adjust to match dataset resolution
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Video format

for clip_idx, (clip_uid, df_gt, df_pred, frame_paths) in enumerate(zip(preds.keys(), dfs_gt, dfs_pred, pred_frames)):
    output_video_path = output_dir / f"{clip_uid}.mp4"

    # Define new video writer per clip
    video_writer = cv2.VideoWriter(str(output_video_path), fourcc, fps, frame_size)

    for frame_idx, (gt_row, pred_row, frame_path) in tqdm(enumerate(zip(df_gt.itertuples(index=False), 
                                                                        df_pred.itertuples(index=False), 
                                                                        frame_paths)), 
                                                           total=len(frame_paths), 
                                                           desc=f"Processing clip {clip_idx+1}/{len(dfs_gt)} ({clip_uid})"):
        img = Image.open(frame_path).convert("RGB")
        draw = ImageDraw.Draw(img)

        # GT Box (Red)
        if not (gt_row.x == -1 and gt_row.y == -1 and gt_row.w == -1 and gt_row.h == -1):
            gt_bbox = [gt_row.x, gt_row.y, gt_row.x + gt_row.w, gt_row.y + gt_row.h]
            draw.rectangle(gt_bbox, outline="red", width=5)

        # Prediction Box (Blue)
        if not (pred_row.x1 == -1 and pred_row.y1 == -1 and pred_row.x2 == -1 and pred_row.y2 == -1):
            pred_bbox = [pred_row.x1, pred_row.y1, pred_row.x2, pred_row.y2]
            draw.rectangle(pred_bbox, outline="blue", width=5)

        # Convert PIL image to OpenCV format
        img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
        img_cv = cv2.resize(img_cv, frame_size)  # Resize to match video output size

        # Write to video
        video_writer.write(img_cv)

    # Release video writer after finishing all frames for this clip
    video_writer.release()
    print(f"Tracking result saved at: {output_video_path}")


In [ ]:
import cv2
import numpy as np
from tqdm import tqdm
from PIL import Image, ImageDraw
from pathlib import Path
import imageio  # GIF 생성용
%cd /data/joohyun7u/project/vq2d-lightning
%load_ext autoreload
%autoreload 2

output_dir = Path("outputs/batch/2025-02-18/84365/trek150/tracking_result")
output_gif_dir = Path("outputs/batch/2025-02-18/84365/trek150/tracking_result_gif")
output_dir.mkdir(parents=True, exist_ok=True)  # Ensure output directory exists
output_gif_dir.mkdir(parents=True, exist_ok=True)  # Ensure output directory exists

fps = 60  # Frame rate for MP4
frame_size = (1280, 720)  # Adjust to match dataset resolution
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Video format

for clip_idx, (clip_uid, df_gt, df_pred, frame_paths) in enumerate(zip(preds.keys(), dfs_gt, dfs_pred, pred_frames)):
    if clip_idx < 50:
        continue
    output_video_path = output_dir / f"{clip_uid}.mp4"
    output_gif_path = output_gif_dir / f"{clip_uid}.gif"

    # Define video writer
    video_writer = cv2.VideoWriter(str(output_video_path), fourcc, fps, frame_size)

    gif_frames = []  # GIF를 위한 프레임 저장 리스트
    frame_skip = fps // 10  # GIF를 위해 초당 10프레임만 사용

    for frame_idx, (gt_row, pred_row, frame_path) in tqdm(enumerate(zip(df_gt.itertuples(index=False), 
                                                                        df_pred.itertuples(index=False), 
                                                                        frame_paths)), 
                                                           total=len(frame_paths), 
                                                           desc=f"Processing clip {clip_idx+1}/{len(dfs_gt)} ({clip_uid})"):
        img = Image.open(frame_path).convert("RGB")
        draw = ImageDraw.Draw(img)

        # GT Box (Red)
        if not (gt_row.x == -1 and gt_row.y == -1 and gt_row.w == -1 and gt_row.h == -1):
            gt_bbox = [gt_row.x, gt_row.y, gt_row.x + gt_row.w, gt_row.y + gt_row.h]
            draw.rectangle(gt_bbox, outline="red", width=3)

        # Prediction Box (Blue)
        if not (pred_row.x1 == -1 and pred_row.y1 == -1 and pred_row.x2 == -1 and pred_row.y2 == -1):
            pred_bbox = [pred_row.x1, pred_row.y1, pred_row.x2, pred_row.y2]
            draw.rectangle(pred_bbox, outline="blue", width=3)

        # Convert PIL image to OpenCV format
        img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
        img_cv = cv2.resize(img_cv, frame_size)  # Resize to match video output size

        # Write to video
        video_writer.write(img_cv)

        # GIF에 추가 (프레임 스킵 적용)
        if frame_idx % frame_skip == 0:
            gif_frames.append(img)

    # Release video writer after finishing all frames for this clip
    video_writer.release()
    print(f"Tracking result saved at: {output_video_path}")

    # GIF 저장 (loop=0: 무한 반복, duration=100ms/frame)
    if gif_frames:
        gif_frames[0].save(output_gif_path, save_all=True, append_images=gif_frames[1:], loop=0, duration=100)
        print(f"GIF saved at: {output_gif_path}")


In [4]:
import numpy as np

def reorder_frames(frame_idxs, frame_interval):
    frame_idxs = np.array(frame_idxs)  # NumPy 배열로 변환

    if frame_interval == 2:
        # 1. 앞으로 진행
        forward = frame_idxs[::frame_interval]
        remaining = np.setdiff1d(frame_idxs, forward, assume_unique=True)  # 선택된 것 제외

        # 2. 뒤로 진행
        backward = remaining[::-1]

        result = np.concatenate([forward, backward])

    elif frame_interval == 3:
        # 1. 앞으로 진행
        forward = frame_idxs[::frame_interval]
        remaining = np.setdiff1d(frame_idxs, forward, assume_unique=True)

        # 2. 뒤로 진행
        backward = remaining[::-1][::frame_interval-1]
        remaining = np.setdiff1d(remaining, backward, assume_unique=True)

        # 3. 다시 앞으로 진행
        third_pass = remaining

        result = np.concatenate([forward, backward, third_pass])

    else:
        raise ValueError("frame_interval은 2 또는 3만 가능합니다.")

    return result

# 예제 실행
frame_idxs = np.array([567, 569, 570, 572, 574, 576, 578, 580, 582, 585, 586, 589, 590,
                       593, 594, 596, 598, 601, 603, 604, 606, 609, 610, 613, 614, 617,
                       619, 620, 623, 625, 626, 629])

frame_interval_2 = reorder_frames(frame_idxs, 2)
frame_interval_3 = reorder_frames(frame_idxs, 3)

print("frame_interval = 2:", frame_idxs)
print("frame_interval = 2:", frame_interval_2)
print("frame_interval = 3:", frame_interval_3)


frame_interval = 2: [567 569 570 572 574 576 578 580 582 585 586 589 590 593 594 596 598 601
 603 604 606 609 610 613 614 617 619 620 623 625 626 629]
frame_interval = 2: [567 570 574 578 582 586 590 594 598 603 606 610 614 619 623 626 629 625
 620 617 613 609 604 601 596 593 589 585 580 576 572 569]
frame_interval = 3: [567 572 578 585 590 596 603 609 614 620 626 629 623 617 610 604 598 593
 586 580 574 569 570 576 582 589 594 601 606 613 619 625]


In [3]:
%cd /data/joohyun7u/project/vq2d-lightning
%load_ext autoreload
%autoreload 2
import json
from pathlib import Path

import torch

import lightning as L

from lightning.pytorch.callbacks import BasePredictionWriter

from ltvu.utils.compute_results import get_final_preds_vq2d, fix_predictions_order, get_final_preds_egotracks
from ltvu.metrics import get_metrics_vq2d, format_metrics_vq2d, get_metrics_egotracks, format_metrics_egotracks, get_metrics_lasot, format_metrics_lasot , get_metrics_trek150, format_metrics_trek150

# def on_predict_epoch_end(self, trainer, pl_module):
    # """Merge segmented features and write to json."""
output_dir = '/data/joohyun7u/project/vq2d-lightning/outputs/batch/2025-02-23/85133/trek150'
p_outdir = Path(output_dir)
p_int_pred = p_outdir / 'intermediate_predictions.pt'
p_pred = p_outdir / 'predictions.json'
p_metrics = p_outdir / 'metrics.json'
p_metrics_log = p_outdir / 'metrics.log'
p_tmp_outdir = p_outdir / 'tmp' / 'test'
        
        
# rank_seg_preds = sorted(rank_seg_preds, key=lambda x: x[:-1])
# torch.save(rank_seg_preds, p_tmp_outdir / f'rank-{trainer.global_rank}.pt')

p_int_pred

# get segmented features
print('Gathering segmented features...')
all_seg_preds = {}
for p_pt in p_tmp_outdir.glob('*.pt'):
    rank_seg_preds = torch.load(p_pt, weights_only=True)
    for qset_uuid, seg_idx, num_segments, pred_output in rank_seg_preds:
        if qset_uuid not in all_seg_preds:
            all_seg_preds[qset_uuid] = [None] * num_segments
        all_seg_preds[qset_uuid][seg_idx] = pred_output

# merge features
print('Merging features...')
qset_preds = {}
for qset_uuid, qset_seg_preds in all_seg_preds.items():
    new_ret_bboxes, new_ret_scores, frame_idxs = [], [], []
    num_segments = len(qset_seg_preds)
    # if self.track_continual:
    seg_pred = qset_seg_preds[0]
    assert seg_pred is not None, f'{qset_uuid}_{0}_{num_segments}'
    frame_idxs = seg_pred['frame_idxs']
    qset_preds[qset_uuid] = {
        'ret_bboxes': seg_pred['ret_bboxes'][:len(frame_idxs)],
        'ret_scores': seg_pred['ret_scores'][:len(frame_idxs)],
    }
    # else:
    #     for seg_idx, seg_pred in enumerate(qset_seg_preds):
    #         assert seg_pred is not None, f'{qset_uuid}_{seg_idx}_{num_segments}'
    #         new_ret_bboxes.append(seg_pred['ret_bboxes'])
    #         new_ret_scores.append(seg_pred['ret_scores'])
    #         frame_idxs.append(seg_pred['frame_idxs'])
    #     frame_idxs = torch.cat(frame_idxs, dim=0)
    #     mask_duplicated = frame_idxs == torch.cat([torch.tensor([-1]), frame_idxs[:-1]])
    #     qset_preds[qset_uuid] = {
    #         'ret_bboxes': torch.cat(new_ret_bboxes, dim=0)[~mask_duplicated],
    #         'ret_scores': torch.cat(new_ret_scores, dim=0)[~mask_duplicated],
    #     }

# save intermediate results
print('Saving intermediate results...')
torch.save(qset_preds, p_int_pred)

# remove temporary files
# for p_tmp in p_tmp_outdir.glob('*'):
#     p_tmp.unlink()
# p_tmp_outdir.rmdir()

# # TODO: Below should be handled by a separate evaluation script

# NOTE: no final results required for Trek150

p_clips_dir = Path("/data/dataset/trek150/TREK-150")
metrics = get_metrics_trek150(p_clips_dir, p_int_pred)
metrics_msg = format_metrics_trek150(metrics)
print(metrics_msg)
p_metrics_log.write_text(metrics_msg + '\n')
json.dump(metrics, p_metrics.open('w'))


/data/joohyun7u/anaconda3/envs/vqlight3/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/data/joohyun7u/project/vq2d-lightning
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Gathering segmented features...
Merging features...
Saving intermediate results...
Trek150 Evaluation
Success Score (SS)                   :  3.602
Normalized Precision Score (NPS)     :  3.661
Generalized Success Robustness (GSR) :  0.198
Prec                                 :  0.113

